# 01 · Benchmark: old DB vs refit DB on N48, RING10, held-out 28 and the panel

Named **configurations** × **DBs** × **seeds** × **case sets**, using persistent loky case workers (hard timeout), MACE-POLAR-1
single points (MACE only measures). With `ENSEMBLE = True` every returned conformer is measured too, which gives the
project targets: **#1 |ΔE| median < 100 kJ/mol, none > 200, #1 usually the lowest returned (top-1), within-ensemble
ρ(score, −ΔE) > 0.6, solve < 20 s**.

**Case sets**
- `n48`: 48 random tmQM complexes (2026-09-29) — excluded from the refit.
- `ring10`: 10 held-out small-donor-ring complexes (2026-09-30) — excluded from the refit.
- `heldout28`: the 40-complex random tmQM sample of 2026-09-23 minus the 12 panel complexes.
- `panel`: 8 DOE aqua (DFT refs), 13 tmQM, 3 legacy guards.

All runs use `angle_bends=True` (ring torsion joints, haptic leans and donor bends are bend DOFs; every number in the
project docs since 09-25 is with bends). Results are cached in `RUNS`; re-running only computes what is missing.
Figures: `02_visualization.ipynb` (reads `all_results.csv`).

**GPU runs (`DEVICE = 'cuda'`, since 2026-10-03):** the solver uses the recorded-step fast path, and all `SEEDS` of a complex go through the solver in one pass (`SEEDS_IN_ONE_PASS`); the result files are still one per seed. `solve_s` is then the seed's share of the pass, so compare it with earlier runs as time per seed, not per solve.


In [ ]:
import sys, os, json, subprocess, time
os.environ['MKL_THREADING_LAYER']="SEQUENTIAL"
from pathlib import Path
import numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jacobik').exists())
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from tools.jdev import bench, env
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)

DBS = {   # historical comparisons require an explicit JACOBIK_OLD_DB
    'old': Path(os.environ.get('JACOBIK_OLD_DB', 'none')),
    'new': Path(os.environ.get('JACOBIK_DB', ROOT / 'data' / 'refit' / 'final' / 'constraints_2026-10-18.json')),
}
DBS = {k: v for k, v in DBS.items() if v.is_file()}
MACE = os.environ.get('JACOBIK_MACE_MODEL_PATH', str(ROOT / 'tools' / 'jdev' / 'MACE-POLAR-1-S.model'))
print(MACE)
os.environ['JACOBIK_MACE_MODEL_PATH'] = MACE
ENSEMBLE = True                    # measure every returned conformer (top-1, rho), in MACE batches
if ENSEMBLE: os.environ['JDEV_ENSEMBLE_DE'] = '1'
MACE_MODE = 'energy-only'           # use 'full' to retain force/stress outputs for diagnostics/Hessian work
MACE_DTYPE = 'float64'              # use 'float64' for precision-sensitive force/Hessian validation
RUNS = ROOT / 'data' / 'benchmark_runs' / time.strftime('%Y-%m-%d')
RUNS.mkdir(parents=True, exist_ok=True)
SEEDS = [11, 12, 13]
# --- devices -----------------------------------------------------------------------------------------
# DEVICE = 'cuda' : solver AND MACE on the GPU; cases run one after another inside one process (JOBS unused).
# DEVICE = None   : CPU solver in JOBS warm worker processes; MACE goes to the GPU when torch can use one
#                   (MACE_DEVICE = 'auto'): ONE model in the driver process evaluates what the workers return.
# Either way the whole benchmark below is ONE process: the DBs and the MACE model load once and stay loaded.
# First time on a machine: `python -m tools.jdev.cuda_check --db <db> --mace <model>` says which is faster.
DEVICE = 'cuda'                      # None | 'cuda'
SOLVER_DEVICE = None               # None (= DEVICE, else cpu) | 'cpu' | 'cuda'
MACE_DEVICE = None                 # None (= DEVICE, else auto) | 'auto' | 'cpu' | 'cuda'
JOBS = 1                           # CPU-solver case processes (each uses SOLVER_WORKERS threads; None = cores / JOBS)
SOLVER_WORKERS = None
SEEDS_IN_ONE_PASS = None           # GPU solver: all SEEDS of a complex in ONE solver pass (None = on when the solver is on CUDA;
                                   # False = one solve per seed). Result files stay one per seed; solve_s is the seed's share.
SETS = ['n48', 'ring10', 'heldout28', 'panel']
# Configurations = solve() keyword arguments (+ '_env' for environment switches of that configuration)
CONFIGS = {
    'cur': {'angle_bends': True},
    # 'donor_bends': {'angle_bends': True, '_env': {'JACOBIK_DONOR_BENDS': '1'}},   # gated off (09-30); retest on the new DB
}
import torch
print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| CUDA available', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('DBs:', {k: str(v) for k, v in DBS.items()})
print(json.dumps(env.fingerprint(DBS.get('new', next(iter(DBS.values()))), Path(MACE)), indent=1))

## 1. Guards: must be green before any number counts

In [ ]:
for label, db in DBS.items():
    r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--ignore=tests/regression', '--tb=short'],
                       env=dict(os.environ, JACOBIK_TEST_DB=str(db)), capture_output=True, text=True)
    print(f'=== {label} ===')
    print(r.stdout)
    if r.stderr: print('STDERR:', r.stderr[:2000])
    assert r.returncode == 0, f'guards failed with the {label} DB'


## 2. Runs: every set × DB × configuration × seed

In [ ]:
# One continuous process for every set x DB x configuration x seed (finished result files are reused).
PATHS = bench.run_all(SETS, DBS, CONFIGS, SEEDS, RUNS, jobs=JOBS, solver_workers=SOLVER_WORKERS, save_xyz=True, verbose=True,
                      mace_mode=MACE_MODE, mace_dtype=MACE_DTYPE,
                      device=DEVICE, solver_device=SOLVER_DEVICE, mace_device=MACE_DEVICE,
                      seeds_in_one_pass=SEEDS_IN_ONE_PASS)
ALL = bench.load(PATHS)
ALL.to_csv(RUNS / 'all_results.csv', index=False)
print(len(ALL), 'rows ->', RUNS / 'all_results.csv')
print(ALL.groupby(['solver_device', 'mace_device'])[['solve_s', 'energy_s']].median().round(2))

## 3. Against the targets

In [ ]:
bench.targets(ALL[ALL.group == 'tmQM'])

In [ ]:
bench.summary(ALL)

## 4. Per case and paired differences (is a change bigger than the seed noise?)

In [ ]:
n48 = ALL[ALL.set == 'n48']
bench.per_case(n48)

In [ ]:
if {'old-cur', 'new-cur'} <= set(ALL.config):
    for s in SETS:
        d = bench.paired_diff(ALL[ALL.set == s], 'old-cur', 'new-cur')
        print(s, '| cases clearly better:', int((d['clear'] & (d['mean'] < 0)).sum()), ' clearly worse:', int((d['clear'] & (d['mean'] > 0)).sum()),
              ' mean paired diff %.1f kJ/mol' % d['mean'].mean())
    bench.paired_diff(n48, 'old-cur', 'new-cur').sort_values('mean')

In [ ]:
# sanity: donors all bound, M-D errors
chk = ALL.groupby(['set', 'config', 'case']).agg(status=('status', lambda s: ','.join(sorted(set(s)))), max_md_err=('max_md_err', 'max')).reset_index()
chk[(chk.status != 'ok') | (chk.max_md_err > 0.5)]